# Build yearly routing destinations

Creates one GeoParquet file for every year from 2015 to 2025. These layers are **not OSM POIs**: they combine GIP motorway exits, public-transport stops and rail stations, higher-education locations, and curated municipal centres.

All input paths are stated explicitly below. Set `OVERWRITE = True` only when you want to regenerate the existing output files.

In [ ]:
from pathlib import Path
import sys
import warnings

import geopandas as gpd
import pandas as pd

# Explicit project paths. Change only if the project is moved.
PROJECT_DIR = Path(r'D:\\CO2_Masterarbeit\\CO2_Masterarbeit')
OUTPUT_DIR = PROJECT_DIR / 'TOOLS' / 'pois'
GIP_DIR = PROJECT_DIR / 'TOOLS' / 'gip-data'
PT_DIR = PROJECT_DIR / 'OGD' / 'Public_Transport'
MUNICIPALITIES_PATH = PROJECT_DIR / 'OGD' / 'Gemeindegrenzen.zip'
EDUCATION_PATH = PROJECT_DIR / 'OGD' / 'Bildungsstandorte.zip'
CENTRES_PATH = OUTPUT_DIR / 'static_routing_destinations.csv'

YEARS = range(2015, 2026)
OVERWRITE = False
CRS_ANALYSIS = 'EPSG:3035'
CRS_ROUTING = 'EPSG:4326'

sys.path.insert(0, str(OUTPUT_DIR))
from destination_builders import gip_motorway_exit_destinations, pt_stop_destinations, rail_station_destinations, yearly_transport_stops


In [ ]:
def read_zip(path):
    return gpd.read_file(f'zip://{path.resolve().as_posix()}')

def study_area():
    municipalities = read_zip(MUNICIPALITIES_PATH).to_crs(CRS_ANALYSIS)
    return gpd.GeoSeries([municipalities.union_all().buffer(5_000)], crs=CRS_ANALYSIS).to_crs(CRS_ROUTING).iloc[0]

def standardise(frame, year):
    columns = ['name', 'ref', 'year', 'poi_type', 'source_poi_id', 'source_file', 'source_schema', 'source_note', 'source_years', 'source_stop_ids', 'imputation_method', 'provenance', 'pt_departures_weekday', 'static_destination', 'train_station_name_regex', 'station_cluster_id', 'station_cluster_member_count', 'station_cluster_member_ids', 'station_cluster_representative_id', 'ramp_cluster_id', 'ramp_component_count', 'off_ramp', 'on_ramp', 'geometry']
    frame = frame.copy()
    for column in columns:
        if column not in frame:
            frame[column] = pd.NA
    frame['year'] = year
    frame['source_years'] = frame['source_years'].fillna(str(year))
    frame['imputation_method'] = frame['imputation_method'].fillna('static_repeated')
    frame['provenance'] = frame['provenance'].fillna(frame['source_schema'])
    source_id = frame['source_poi_id'].astype('string').fillna(pd.Series(frame.index, index=frame.index).astype('string'))
    frame['poi_id'] = frame['poi_type'].astype('string') + '_' + str(year) + '_' + source_id.str.replace(r'[^0-9A-Za-z_]+', '_', regex=True).str.strip('_')
    return gpd.GeoDataFrame(frame[columns + ['poi_id']], geometry='geometry', crs=frame.crs)


In [ ]:
def static_destinations(year, mask):
    education = read_zip(EDUCATION_PATH).to_crs(CRS_ROUTING)
    higher_ed = education[education[['EINRICHTUN', 'TYP_LANG', 'TYP_DETAIL']].isin({'Universität', 'Fachhochschule', 'Pädagogische Hochschule'}).any(axis=1) & education.geometry.within(mask)].copy()
    higher_ed = higher_ed.assign(name=higher_ed['NAME'], ref=higher_ed['KENNZAHL'].astype('string'), poi_type='higher_education', source_poi_id=higher_ed['OBJECTID'].astype('string'), source_file=EDUCATION_PATH.name, source_schema='ogd_bildungsstandorte_higher_education', source_note='static_ogd_destination', static_destination=True)

    municipalities = read_zip(MUNICIPALITIES_PATH).to_crs(CRS_ANALYSIS)
    centres = pd.read_csv(CENTRES_PATH).merge(municipalities[['GEMNR', 'GEMNAM', 'geometry']], left_on='municipality_name', right_on='GEMNAM', how='left', validate='many_to_one')
    centres = gpd.GeoDataFrame(centres, geometry='geometry', crs=CRS_ANALYSIS)
    centres['geometry'] = centres.geometry.representative_point()
    centres = centres.to_crs(CRS_ROUTING)[lambda x: x.geometry.within(mask)].copy()
    centres = centres.assign(ref=centres['GEMNR'].astype('string'), source_poi_id=centres['poi_type'].astype('string') + '_' + centres['municipality_name'].astype('string'), source_file=CENTRES_PATH.name, source_schema='curated_static_municipality_representative_points', static_destination=True)
    return pd.concat([higher_ed, centres], ignore_index=True)

mask = study_area()
for year in YEARS:
    output = OUTPUT_DIR / f'austria-{year}-pois.geoparquet'
    if output.exists() and not OVERWRITE:
        print(f'Skipped {output.name}')
        continue
    stops = yearly_transport_stops(PROJECT_DIR, year)
    stops = stops[stops.geometry.within(mask)].copy()
    pt_stops = pt_stop_destinations(stops)
    rail_stations, _ = rail_station_destinations(stops)
    exits = gip_motorway_exit_destinations(year, GIP_DIR / f'{year}.osm.pbf', mask)
    destinations = gpd.GeoDataFrame(pd.concat([pt_stops, rail_stations, exits, static_destinations(year, mask)], ignore_index=True), geometry='geometry', crs=CRS_ROUTING).to_crs(CRS_ANALYSIS)
    standardise(destinations, year).to_parquet(output, index=False)
    print(f'Wrote {len(destinations):,} destinations to {output.name}')
